In [1]:
!pip install pandas pyarrow boto3 --quiet


In [2]:
import io, os, json, time, hashlib, shutil
from datetime import datetime, date, timezone
import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError

ENDPOINT = "http://minio:9000"
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
)

AUJOURDHUI = date.today().isoformat()
journal = []  # une ligne par transformation (traçabilité)

def log_transfo(etape, source, cible, lues, ecrites, rejetees, regles):
    journal.append({
        "horodatage": datetime.now().isoformat(timespec="seconds"),
        "etape": etape, "source": source, "cible": cible,
        "lignes_lues": lues, "lignes_ecrites": ecrites,
        "lignes_rejetees": rejetees, "regles": regles,
    })

def dernier_objet(bucket, prefixe):
    """Objet le plus récent sous un préfixe (métadonnées S3)."""
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixe)
    objets = [o for page in pages for o in page.get("Contents", [])]
    if not objets:
        raise FileNotFoundError("Aucun objet sous " + bucket + "/" + prefixe)
    return max(objets, key=lambda o: o["LastModified"])

def lire_csv(bucket, cle, **options):
    corps = s3.get_object(Bucket=bucket, Key=cle)["Body"].read()
    # utf-8-sig : tolère un éventuel BOM ajouté par PowerShell
    return pd.read_csv(io.BytesIO(corps), encoding="utf-8-sig", **options)

def lire_parquet(bucket, cle):
    corps = s3.get_object(Bucket=bucket, Key=cle)["Body"].read()
    return pd.read_parquet(io.BytesIO(corps))

def ecrire_parquet(df, bucket, cle):
    tampon = io.BytesIO()
    df.to_parquet(tampon, index=False)
    tampon.seek(0)
    s3.upload_fileobj(tampon, bucket, cle)

def ecrire_csv(df, bucket, cle):
    s3.put_object(Bucket=bucket, Key=cle, Body=df.to_csv(index=False).encode("utf-8"))

def lister(bucket, prefixe=""):
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixe)
    return [o["Key"] for page in pages for o in page.get("Contents", [])]

print("Environnement prêt -", AUJOURDHUI)


Environnement prêt - 2026-11-19


In [3]:
ATTENDU = {
    "raw": ["postgres_commandes/", "api_clients/", "fichiers_stock/"],
    "trusted": [], "curated": [],
}
manque = []
for bucket, prefixes in ATTENDU.items():
    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError:
        manque.append("bucket " + bucket)
        continue
    for prefixe in prefixes:
        if not lister(bucket, prefixe):
            manque.append(bucket + "/" + prefixe)
print("Prérequis OK" if not manque else "À reprendre (TP 1 ou TP 2) : " + ", ".join(manque))


Prérequis OK


In [4]:
cle = "fichiers_commandes_import/" + AUJOURDHUI.replace("-", "/") + "/commandes_import.csv"
s3.upload_file("commandes_import.csv", "raw", cle)
print("Déposé dans raw :", cle)


Déposé dans raw : fichiers_commandes_import/2026/11/19/commandes_import.csv


In [5]:
SOURCES = {
    "commandes PostgreSQL": "postgres_commandes/",
    "commandes import CSV": "fichiers_commandes_import/",
    "stock": "fichiers_stock/",
}
for nom, prefixe in SOURCES.items():
    obj = dernier_objet("raw", prefixe)
    df = lire_csv("raw", obj["Key"])
    print("==", nom, "|", obj["Key"], "|", len(df), "lignes")
    print("   colonnes :", list(df.columns))
    print("   valeurs vides :", df.isna().sum().to_dict())
    print("   lignes dupliquées :", int(df.duplicated().sum()))


== commandes PostgreSQL | postgres_commandes/2026/10/30/commandes.csv | 4 lignes
   colonnes : ['id', 'client_id', 'produit', 'quantite', 'prix_unitaire', 'date_commande']
   valeurs vides : {'id': 0, 'client_id': 0, 'produit': 0, 'quantite': 0, 'prix_unitaire': 0, 'date_commande': 0}
   lignes dupliquées : 0
== commandes import CSV | fichiers_commandes_import/2026/11/19/commandes_import.csv | 8 lignes
   colonnes : ['id', 'client_id', 'produit', 'quantite', 'prix_unitaire', 'date_commande']
   valeurs vides : {'id': 0, 'client_id': 1, 'produit': 0, 'quantite': 0, 'prix_unitaire': 1, 'date_commande': 0}
   lignes dupliquées : 1
== stock | fichiers_stock/2026/10/30/stock.csv | 4 lignes
   colonnes : ['produit', 'stock', 'entrepot']
   valeurs vides : {'produit': 0, 'stock': 0, 'entrepot': 0}
   lignes dupliquées : 0


In [6]:
obj = dernier_objet("raw", "api_clients/")
utilisateurs = json.loads(s3.get_object(Bucket="raw", Key=obj["Key"])["Body"].read())
brut_clients = pd.json_normalize(utilisateurs)  # aplatit address.city, company.name...

clients = pd.DataFrame({
    "id_client": brut_clients["id"].astype(int),
    "nom": brut_clients["name"].str.strip(),
    "email": brut_clients["email"].str.strip().str.lower(),
    "telephone": brut_clients["phone"].str.strip(),
    "ville": brut_clients["address.city"].str.strip(),
}).drop_duplicates(subset="id_client")

cle = "clients/date_traitement=" + AUJOURDHUI + "/clients.parquet"
ecrire_parquet(clients, "trusted", cle)
log_transfo("raw->trusted", obj["Key"], cle, len(brut_clients), len(clients), 0,
            "aplatissement JSON, strip, email en minuscules")
print(len(clients), "clients écrits dans trusted")


10 clients écrits dans trusted


In [7]:
obj = dernier_objet("raw", "fichiers_stock/")
brut_stock = lire_csv("raw", obj["Key"])

stock = brut_stock.copy()
stock["produit"] = stock["produit"].str.strip().str.capitalize()  # R01
stock["stock"] = pd.to_numeric(stock["stock"], errors="coerce").astype("Int64")
stock = stock.drop_duplicates(subset="produit")

cle = "stock/date_traitement=" + AUJOURDHUI + "/stock.parquet"
ecrire_parquet(stock, "trusted", cle)
log_transfo("raw->trusted", obj["Key"], cle, len(brut_stock), len(stock), 0, "R01, typage stock")
print(stock)


               produit  stock   entrepot
0  Ordinateur portable     42      Paris
1      Souris sans fil    310       Lyon
2    Clavier mecanique    128       Lyon
3      Ecran 24 pouces     57  Marseille


In [8]:
email_valide = clients["email"].str.contains("@", regex=False)
print("Emails invalides :", int((~email_valide).sum()))
print(clients.loc[~email_valide, ["id_client", "email"]])


Emails invalides : 0
Empty DataFrame
Columns: [id_client, email]
Index: []


In [9]:
COLONNES = ["id", "client_id", "produit", "quantite", "prix_unitaire", "date_commande"]

def lire_commandes(prefixe, source):
    obj = dernier_objet("raw", prefixe)
    df = lire_csv("raw", obj["Key"], dtype=str)[COLONNES].copy()  # tout en texte au départ
    df["source"] = source
    return df, obj

pg, obj_pg = lire_commandes("postgres_commandes/", "postgres")
imp, obj_imp = lire_commandes("fichiers_commandes_import/", "import_csv")
brut = pd.concat([pg, imp], ignore_index=True)
n_lues = len(brut)

df = pd.DataFrame({
    "id": pd.to_numeric(brut["id"], errors="coerce"),
    "client_id": pd.to_numeric(brut["client_id"], errors="coerce"),
    "produit": brut["produit"].str.strip().str.capitalize(),  # R01
    "quantite": pd.to_numeric(brut["quantite"], errors="coerce"),
    "prix_unitaire": pd.to_numeric(  # R02
        brut["prix_unitaire"].str.strip().str.replace(",", ".", regex=False),
        errors="coerce"),
    "source": brut["source"],
})

# R05 : deux formats de date acceptés, AAAA-MM-JJ puis JJ/MM/AAAA
texte_date = brut["date_commande"].str.strip()
iso = pd.to_datetime(texte_date, format="%Y-%m-%d", errors="coerce")
fr = pd.to_datetime(texte_date, format="%d/%m/%Y", errors="coerce")
df["date_commande"] = iso.fillna(fr)

# R06 : doublons exacts écartés (et comptés)
avant = len(df)
df = df.drop_duplicates(subset=COLONNES).reset_index(drop=True)
n_doublons = avant - len(df)
print(n_lues, "lignes lues,", n_doublons, "doublon(s) écarté(s),", len(df), "lignes à contrôler")


12 lignes lues, 1 doublon(s) écarté(s), 11 lignes à contrôler


In [10]:
print(df.dtypes)
print()
print(df.to_string())


id                        int64
client_id               float64
produit                  object
quantite                  int64
prix_unitaire           float64
source                   object
date_commande    datetime64[ns]
dtype: object

     id  client_id              produit  quantite  prix_unitaire      source date_commande
0     1        1.0  Ordinateur portable         2         899.99    postgres    2026-09-05
1     2        2.0      Souris sans fil         5          19.99    postgres    2026-09-06
2     3        3.0    Clavier mecanique         1          59.99    postgres    2026-09-07
3     4        1.0      Ecran 24 pouces         1         179.99    postgres    2026-09-08
4   101        1.0      Souris sans fil         3          19.99  import_csv    2026-08-28
5   102        2.0    Clavier mecanique         1          59.99  import_csv    2026-09-11
6   103        NaN      Ecran 24 pouces         2         179.99  import_csv    2026-09-12
7   104        3.0      Souris sa

In [11]:
ids_clients = set(clients["id_client"])

# R03 : une règle de rejet = un motif ; le premier motif rencontré est conservé
regles_rejet = [
    ("client_id manquant", df["client_id"].isna()),
    ("prix manquant, non numérique ou <= 0", df["prix_unitaire"].isna() | (df["prix_unitaire"] <= 0)),
    ("quantité manquante ou <= 0", df["quantite"].isna() | (df["quantite"] <= 0)),
    ("date manquante ou invalide", df["date_commande"].isna()),
    ("client inconnu", df["client_id"].notna() & ~df["client_id"].isin(ids_clients)),
]
df["motif_rejet"] = None
for motif, condition in regles_rejet:
    df.loc[condition & df["motif_rejet"].isna(), "motif_rejet"] = motif

controles = []
def controle(nom, dimension, valeur, seuil):
    controles.append({
        "controle": nom, "dimension": dimension,
        "valeur": round(float(valeur), 4), "seuil": seuil,
        "statut": "OK" if valeur <= seuil else "ALERTE",
    })

controle("taux de client_id vide", "complétude", df["client_id"].isna().mean(), 0.05)
controle("taux de prix invalides", "validité", regles_rejet[1][1].mean(), 0.05)
controle("taux de quantités invalides", "validité", regles_rejet[2][1].mean(), 0.05)
controle("taux de dates invalides", "validité", regles_rejet[3][1].mean(), 0.05)
controle("taux de doublons d'id", "unicité", df["id"].duplicated().mean(), 0.0)
controle("taux de clients inconnus", "cohérence", regles_rejet[4][1].mean(), 0.0)
age_heures = (datetime.now(timezone.utc) - obj_pg["LastModified"]).total_seconds() / 3600
controle("âge de l'ingestion PostgreSQL (h)", "fraîcheur", age_heures, 72)

rapport = pd.DataFrame(controles)
print(rapport.to_string(index=False))


                         controle  dimension   valeur  seuil statut
           taux de client_id vide complétude   0.0909   0.05 ALERTE
           taux de prix invalides   validité   0.0909   0.05 ALERTE
      taux de quantités invalides   validité   0.0909   0.05 ALERTE
          taux de dates invalides   validité   0.0000   0.05     OK
            taux de doublons d'id    unicité   0.0000   0.00     OK
         taux de clients inconnus  cohérence   0.0909   0.00 ALERTE
âge de l'ingestion PostgreSQL (h)  fraîcheur 478.0810  72.00 ALERTE


In [14]:
rejets = df[df["motif_rejet"].notna()].copy()
valides = df[df["motif_rejet"].isna()].drop(columns="motif_rejet").astype(
    {"id": "int64", "client_id": "int64", "quantite": "int64"})

taux_rejet = len(rejets) / len(df)
SEUIL_BLOCAGE = 0.5
if taux_rejet > SEUIL_BLOCAGE:
    raise RuntimeError("Lot bloqué : " + format(taux_rejet, ".0%") + " de lignes rejetées")

suffixe = "date_traitement=" + AUJOURDHUI + "/"
ecrire_parquet(valides, "trusted", "commandes/" + suffixe + "commandes.parquet")
ecrire_csv(rejets, "trusted", "_rejets/commandes/" + suffixe + "rejets.csv")
ecrire_csv(rapport, "trusted", "_qualite/commandes/" + suffixe + "rapport_qualite.csv")
log_transfo("raw->trusted", "postgres_commandes + fichiers_commandes_import",
            "trusted/commandes/" + suffixe, n_lues, len(valides), len(rejets),
            "R01 R02 R03 R05 R06 ; doublons écartés : " + str(n_doublons))

print(len(valides), "lignes valides,", len(rejets), "en quarantaine (" + format(taux_rejet, ".0%") + ")")
print(rejets[["id", "client_id", "motif_rejet"]].to_string(index=False))


7 lignes valides, 4 en quarantaine (36%)
 id  client_id                          motif_rejet
103        NaN                   client_id manquant
104        3.0 prix manquant, non numérique ou <= 0
105        1.0           quantité manquante ou <= 0
107       99.0                       client inconnu


In [15]:
controle("taux de quantités > 1000", "validité", (df["quantite"] > 1000).mean(), 0.0)
print(pd.DataFrame(controles).tail(1).to_string(index=False))


                controle dimension  valeur  seuil statut
taux de quantités > 1000  validité     0.0    0.0     OK


In [16]:
CLASSIFICATION = {
    "nom": "sensible",
    "email": "sensible",
    "telephone": "sensible",
    "ville": "interne",
    "id_client": "interne",
    "produit": "public",
    "prix_unitaire": "interne",
}
for colonne, niveau in CLASSIFICATION.items():
    print(colonne.ljust(15), niveau)


nom             sensible
email           sensible
telephone       sensible
ville           interne
id_client       interne
produit         public
prix_unitaire   interne


In [17]:
SEL = os.environ.get("SEL_PSEUDO", "sel-de-formation")  # en production : coffre à secrets

def pseudonymiser(valeur):
    texte = SEL + str(valeur).strip().lower()
    return hashlib.sha256(texte.encode("utf-8")).hexdigest()[:16]

dim_client = clients.assign(email_pseudo=clients["email"].map(pseudonymiser))[
    ["id_client", "ville", "email_pseudo"]]

# Contrôle : aucune donnée sensible directe ne doit rester dans curated
assert not {"nom", "email", "telephone"} & set(dim_client.columns)

ecrire_parquet(dim_client, "curated", "dim_client/" + suffixe + "dim_client.parquet")
ecrire_parquet(stock, "curated", "dim_stock/" + suffixe + "dim_stock.parquet")
log_transfo("trusted->curated", "trusted/clients", "curated/dim_client/" + suffixe,
            len(clients), len(dim_client), 0, "R04 pseudonymisation de l'email")
print(dim_client.head())


   id_client          ville      email_pseudo
0          1    Gwenborough  a50062e765b1c22d
1          2    Wisokyburgh  ccfe7371302f0275
2          3  McKenziehaven  a9fd49f49e467d47
3          4    South Elvis  ce94ee38c1afb206
4          5     Roscoeview  f2f3acfbb22a6ee5


In [18]:
fait_ventes = valides.merge(dim_client[["id_client", "ville"]],
                            left_on="client_id", right_on="id_client", how="left")
fait_ventes = fait_ventes.drop(columns="id_client").rename(columns={"id": "id_commande"})
fait_ventes["montant"] = (fait_ventes["quantite"] * fait_ventes["prix_unitaire"]).round(2)
fait_ventes["annee"] = fait_ventes["date_commande"].dt.strftime("%Y")
fait_ventes["mois"] = fait_ventes["date_commande"].dt.strftime("%m")

# Une partition par année et par mois
for (annee, mois), partie in fait_ventes.groupby(["annee", "mois"]):
    cle = "fait_ventes/annee=" + annee + "/mois=" + mois + "/part-0001.parquet"
    ecrire_parquet(partie, "curated", cle)
    print("écrit :", cle, "(" + str(len(partie)) + " lignes)")

agg = fait_ventes.groupby("produit", as_index=False).agg(
    nb_commandes=("id_commande", "count"),
    quantite_totale=("quantite", "sum"),
    chiffre_affaires=("montant", "sum"))
ecrire_parquet(agg, "curated", "agg_ventes_produit/" + suffixe + "agg.parquet")
log_transfo("trusted->curated", "trusted/commandes", "curated/fait_ventes/",
            len(valides), len(fait_ventes), 0, "jointure client, montant, partition annee/mois")
print(agg)


écrit : fait_ventes/annee=2026/mois=08/part-0001.parquet (2 lignes)
écrit : fait_ventes/annee=2026/mois=09/part-0001.parquet (5 lignes)
               produit  nb_commandes  quantite_totale  chiffre_affaires
0    Clavier mecanique             2                2            119.98
1      Ecran 24 pouces             2                2            359.98
2  Ordinateur portable             1                2           1799.98
3      Souris sans fil             2                8            159.92


In [19]:
cles = lister("curated", "fait_ventes/")
relu = pd.concat([lire_parquet("curated", c) for c in cles], ignore_index=True)
print(len(relu), "lignes relues depuis", len(cles), "fichier(s) Parquet")
print(relu.dtypes)
print(relu.groupby("mois")["montant"].sum())


7 lignes relues depuis 2 fichier(s) Parquet
id_commande               int64
client_id                 int64
produit                  object
quantite                  int64
prix_unitaire           float64
source                   object
date_commande    datetime64[ns]
ville                    object
montant                 float64
annee                    object
mois                     object
dtype: object
mois
08     239.96
09    2199.90
Name: montant, dtype: float64


In [20]:
ca_ville = fait_ventes.groupby("ville", as_index=False)["montant"].sum().sort_values("montant", ascending=False)
print(ca_ville)


           ville  montant
0    Gwenborough  2039.94
2    South Elvis   179.99
3    Wisokyburgh   159.94
1  McKenziehaven    59.99


In [21]:
for cle in lister("curated"):
    if cle.endswith(".parquet"):
        print(cle, "->", list(lire_parquet("curated", cle).columns))


agg_ventes_produit/date_traitement=2026-11-19/agg.parquet -> ['produit', 'nb_commandes', 'quantite_totale', 'chiffre_affaires']
dim_client/date_traitement=2026-11-19/dim_client.parquet -> ['id_client', 'ville', 'email_pseudo']
dim_stock/date_traitement=2026-11-19/dim_stock.parquet -> ['produit', 'stock', 'entrepot']
fait_ventes/annee=2026/mois=08/part-0001.parquet -> ['id_commande', 'client_id', 'produit', 'quantite', 'prix_unitaire', 'source', 'date_commande', 'ville', 'montant', 'annee', 'mois']
fait_ventes/annee=2026/mois=09/part-0001.parquet -> ['id_commande', 'client_id', 'produit', 'quantite', 'prix_unitaire', 'source', 'date_commande', 'ville', 'montant', 'annee', 'mois']


In [22]:
s3_analyste = boto3.client(
    "s3", endpoint_url=ENDPOINT,
    aws_access_key_id="analyste", aws_secret_access_key="analyste-mdp-2026")

def tester(libelle, action):
    try:
        action()
        print("AUTORISÉ :", libelle)
    except ClientError as erreur:
        print("REFUSÉ   :", libelle, "(" + erreur.response["Error"]["Code"] + ")")

tester("lire curated/agg_ventes_produit",
       lambda: s3_analyste.list_objects_v2(Bucket="curated", Prefix="agg_ventes_produit/"))
tester("lister le bucket raw",
       lambda: s3_analyste.list_objects_v2(Bucket="raw"))
tester("écrire dans curated",
       lambda: s3_analyste.put_object(Bucket="curated", Key="essai.txt", Body=b"x"))


AUTORISÉ : lire curated/agg_ventes_produit
REFUSÉ   : lister le bucket raw (AccessDenied)
REFUSÉ   : écrire dans curated (AccessDenied)


In [23]:
rng = np.random.default_rng(42)
N = 500_000
produits = np.array(["Ordinateur portable", "Souris sans fil", "Clavier mecanique", "Ecran 24 pouces"])
prix_ref = np.array([899.99, 19.99, 59.99, 179.99])
choix = rng.integers(0, 4, N)
jours = pd.to_timedelta(rng.integers(0, 365, N), unit="D")
gros = pd.DataFrame({
    "id": np.arange(N),
    "client_id": rng.integers(1, 11, N),
    "produit": produits[choix],
    "quantite": rng.integers(1, 6, N),
    "prix_unitaire": prix_ref[choix],
    "date_commande": pd.Timestamp("2026-01-01") + jours,
    "commentaire": rng.choice(["livraison standard a domicile", "retrait en magasin",
                               "livraison express", "cadeau emballage"], N),
})
gros["montant"] = (gros["quantite"] * gros["prix_unitaire"]).round(2)
gros["mois"] = gros["date_commande"].dt.month  # entier 1..12
print(len(gros), "lignes,", gros.shape[1], "colonnes")


500000 lignes, 9 colonnes


In [25]:
def chrono(fonction):
    debut = time.perf_counter()
    resultat = fonction()
    return resultat, time.perf_counter() - debut

def taille_mo(chemin):
    if os.path.isdir(chemin):
        total = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(chemin) for f in fs)
    else:
        total = os.path.getsize(chemin)
    return round(total / 1_048_576, 2)

shutil.rmtree("bench_part", ignore_errors=True)
_, ecr_csv = chrono(lambda: gros.to_csv("bench.csv", index=False))
_, ecr_pq = chrono(lambda: gros.to_parquet("bench.parquet", index=False))
_, ecr_pp = chrono(lambda: gros.to_parquet("bench_part", partition_cols=["mois"], index=False))
_, lec_csv = chrono(lambda: pd.read_csv("bench.csv", usecols=["montant"]))
_, lec_pq = chrono(lambda: pd.read_parquet("bench.parquet", columns=["montant"]))
_, lec_pp = chrono(lambda: pd.read_parquet(
    "bench_part", columns=["montant"], filters=[("mois", "=", 9)]))

resultats = pd.DataFrame([
    {"format": "CSV", "taille_Mo": taille_mo("bench.csv"),
     "ecriture_s": ecr_csv, "lecture_1_colonne_s": lec_csv},
    {"format": "Parquet", "taille_Mo": taille_mo("bench.parquet"),
     "ecriture_s": ecr_pq, "lecture_1_colonne_s": lec_pq},
    {"format": "Parquet partitionné (mois=09)", "taille_Mo": taille_mo("bench_part"),
     "ecriture_s": ecr_pp, "lecture_1_colonne_s": lec_pp},
]).round(3)
resultats["debit_lignes_par_s"] = (N / resultats["ecriture_s"]).round(0)
print(resultats.to_string(index=False))


                       format  taille_Mo  ecriture_s  lecture_1_colonne_s  debit_lignes_par_s
                          CSV      36.25       2.723                0.692            183621.0
                      Parquet       4.04       0.422                0.021           1184834.0
Parquet partitionné (mois=09)       4.64       0.674                0.178            741840.0


In [26]:
_, up_csv = chrono(lambda: s3.upload_file("bench.csv", "trusted", "_benchmark/bench.csv"))
_, up_pq = chrono(lambda: s3.upload_file("bench.parquet", "trusted", "_benchmark/bench.parquet"))
_, dl_csv = chrono(lambda: s3.download_file("trusted", "_benchmark/bench.csv", "bench_dl.csv"))
_, dl_pq = chrono(lambda: s3.download_file("trusted", "_benchmark/bench.parquet", "bench_dl.parquet"))

transfert = pd.DataFrame([
    {"format": "CSV", "envoi_s": up_csv, "lecture_s": dl_csv},
    {"format": "Parquet", "envoi_s": up_pq, "lecture_s": dl_pq},
]).round(3)
print(transfert.to_string(index=False))

resultats.to_csv("kpi_pipeline.csv", index=False)
transfert.to_csv("kpi_transfert.csv", index=False)


 format  envoi_s  lecture_s
    CSV    0.812      1.676
Parquet    0.192      0.314


In [27]:
for codec in ["snappy", "gzip", "zstd"]:
    _, duree = chrono(lambda: gros.to_parquet("bench_" + codec + ".parquet", compression=codec, index=False))
    print(codec.ljust(8), taille_mo("bench_" + codec + ".parquet"), "Mo", round(duree, 2), "s")


snappy   4.04 Mo 0.28 s
gzip     2.74 Mo 0.56 s
zstd     2.54 Mo 0.25 s


In [28]:
import glob
for fichier in glob.glob("bench*.csv") + glob.glob("bench*.parquet"):
    os.remove(fichier)
shutil.rmtree("bench_part", ignore_errors=True)
del gros
print("Fichiers de benchmark supprimés du dossier de travail")


Fichiers de benchmark supprimés du dossier de travail


In [29]:
journal_df = pd.DataFrame(journal)
journal_df.to_csv("journal_transformation.csv", index=False, encoding="utf-8")
s3.upload_file("journal_transformation.csv", "trusted",
               "_journal/date_traitement=" + AUJOURDHUI + "/journal_transformation.csv")
print(journal_df[["etape", "cible", "lignes_lues", "lignes_ecrites", "lignes_rejetees"]].to_string(index=False))


           etape                                              cible  lignes_lues  lignes_ecrites  lignes_rejetees
    raw->trusted clients/date_traitement=2026-11-19/clients.parquet           10              10                0
    raw->trusted     stock/date_traitement=2026-11-19/stock.parquet            4               4                0
    raw->trusted      trusted/commandes/date_traitement=2026-11-19/           12               7                4
    raw->trusted      trusted/commandes/date_traitement=2026-11-19/           12               7                4
trusted->curated     curated/dim_client/date_traitement=2026-11-19/           10              10                0
trusted->curated                               curated/fait_ventes/            7               7                0
